# Review a trained model

Run after `ribbon train` (and ideally `ribbon evaluate <run>`). Three questions:

1. **Did training go well?** Learning curves: is validation improving, or has it stalled while
   training accuracy keeps climbing (over-fitting)?
2. **Which classes get confused?** Confusion matrix.
3. **Why?** Look at the actual misclassified images, most-confident mistakes first. These are
   usually label errors, ambiguous photos, or a missing class, so fix the data, not the model.

In [ ]:
import json
import os

import matplotlib.pyplot as plt
import pandas as pd

from ribbon_classifier.config import load_config
from ribbon_classifier.dataset import load_image
from ribbon_classifier.evaluate import evaluate_run

config = load_config(os.environ.get("RIBBON_CONFIG"))
models_dir = config.path(config.paths.models_dir)
runs = sorted(p for p in models_dir.iterdir() if (p / "meta.json").exists())
if not runs:
    raise FileNotFoundError("No trained runs in models/: run `ribbon train` first.")
run_dir = runs[-1]  # newest; or set explicitly, e.g. models_dir / "20260927-101500_convnext_tiny"
meta = json.loads((run_dir / "meta.json").read_text(encoding="utf-8"))
print(run_dir.name, "| best epoch", meta["best_epoch"], "| val macro-F1", meta["val_macro_f1"])
print("classes:", meta["classes"])

## Learning curves

* Train accuracy ≫ validation accuracy and the gap keeps growing → over-fitting: more
  augmentation, fewer epochs, more data, or freeze more of the backbone.
* Both low and flat → under-fitting or a data problem: check labels and the learning rate.
* The first `head_only_epochs` train only the new head, so a jump when the backbone unfreezes
  is expected.

In [ ]:
history = pd.read_csv(run_dir / "history.csv")
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
history.plot(x="epoch", y="train_loss", ax=ax1, title="Training loss", legend=False)
history.plot(x="epoch", y=["train_acc", "val_acc", "val_macro_f1"], ax=ax2, title="Accuracy / F1", ylim=(0, 1.05))
ax2.axvline(meta["best_epoch"], color="gray", linestyle="--", label="best")
plt.show()

## Confusion matrix and per-class scores

Uses the test split if it has been evaluated; set `SPLIT = "val"` while you are still iterating
so the test set stays untouched until the end.

In [ ]:
SPLIT = "test"
result = evaluate_run(config, run_dir, split=SPLIT)  # writes eval_*.json, confusion_*.png, errors_*.csv
print(f"{SPLIT}: accuracy {result['accuracy']:.3f}, macro-F1 {result['macro_f1']:.3f}")
per_class = pd.DataFrame(result["per_class"]).T.loc[meta["classes"], ["precision", "recall", "f1-score", "support"]]
display(per_class.round(3))
plt.figure(figsize=(7, 7))
plt.imshow(plt.imread(run_dir / f"confusion_{SPLIT}.png"))
plt.axis("off")
plt.show()

## The mistakes themselves

In [ ]:
mistakes = pd.read_csv(run_dir / f"errors_{SPLIT}.csv")
print(len(mistakes), "misclassified")
data_dir = config.path(config.paths.data_dir)
shown = mistakes.head(12)
if len(shown):
    cols = 4
    rows = (len(shown) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(3.2 * cols, 3.4 * rows), squeeze=False)
    for ax in axes.flat:
        ax.axis("off")
    for ax, item in zip(axes.flat, shown.itertuples(), strict=False):
        ax.imshow(load_image(data_dir / item.image_path))
        ax.set_title(f"true: {item.label}\npred: {item.predicted} ({item.confidence:.2f})", fontsize=9)
    plt.tight_layout()
    plt.show()